In [1]:
import kosh
from sina.utils import DataRange, has_all, has_any, all_in, any_in
import os
import numpy as np
import random

# Simulation Workflow

This notebook will take a user through a simple simulation workflow with an inner loop (single simulations) and outer loop (grouping simulations into an ensemble).

## Create Overall Store

This will house all the simulation data

In [2]:
store_path = 'my_store.sqlite' # "/usr/workspace/group_dir/group.sqlite"
try:
    os.remove(store_path)
except:
    pass
store = kosh.connect(store_path)

## Inner Loop

This inner loop contains the data for the individual simulations

### Create Dataset for a Simulation

We create our first dataset and add `metadata` to it which can be used to filter datasets using the `store.find()` method later on. These are single value items and are dataset attributes.

See [Example_00_Open_Store_And_Add_Datasets.ipynb](Example_00_Open_Store_And_Add_Datasets.ipynb) for more examples on how to add data. 

See [Example_04_Schemas.ipynb](Example_04_Schemas.ipynb) for examples on how to set up a schema for the metadata.

In [3]:
# If your code outputs a sina json formatted file.
dataset = store.import_dataset("sina_curve_rec.json",
                               match_attributes=['id'])[0]

# important data that can be used in store.find() method later
my_metadata ={'param4': 1,
              'param5': 'test',
              'param6': 3.14}

for key, val in my_metadata.items():
        setattr(dataset, key, val)

# You can also create a dataset from scratch
# dataset = store.create(name='My Example Dataset', # name to find dataset later on
#                        metadata=my_metadata)

# The `metadata` will be in the attributes
# The sina record already has some metadata pre-populated
# It also has some associated data which will be discussed in the upcoming sections
print(dataset)

/g/g20/moreno45/Projects/ASCAML/kosh/kosh/store.py:888: UserWarning: When searching by id use id_pool
  warnings.warn("When searching by id use id_pool")


KOSH DATASET
	id: obj1
	name: ???
	creator: ???

--- Attributes ---
	param1: 1
	param2: 2
	param3: 3.3
	param4: 1
	param5: test
	param6: 3.14
--- Associated Data (2)---
	Mime_type: image/png
		foo.png ( obj1 )
	Mime_type: sina/curve
		internal ( timeplot_1 )
--- Ensembles (0)---
	[]
--- Ensemble Attributes ---
--- Alias Feature Dictionary ---


### Adding Data to a Dataset

You can also add time history data to the dataset. Each time history data is grouped into `curve_sets` with one `independent` curve and one or more `dependent` curves.

See [Example_01_Add_Data_To_Datasets.ipynb](Example_01_Add_Data_To_Datasets.ipynb) for more examples on how to add curve sets.

In [4]:
# dataset.add_curve(np.array(my_array).tolist(), 'my_time_series', 'x_pos')
dataset.add_curve([1,2,3,4], "my_curves", "time")
dataset.add_curve([2.3, 3.4, 5.6, 7.8], "my_curves", "some_variable")
dataset.add_curve([3, 4,5], "my_other_curves", "time")

# Adding a single value to the metadata
some_variable_mean = float(np.mean(dataset['my_curves/some_variable'][:]))
setattr(dataset, 'some_variable_mean', some_variable_mean)

# The `curve_sets` are located in the associated data section as `mime_type="sina/curve"`.
print(dataset)

KOSH DATASET
	id: obj1
	name: ???
	creator: ???

--- Attributes ---
	param1: 1
	param2: 2
	param3: 3.3
	param4: 1
	param5: test
	param6: 3.14
	some_variable_mean: 4.7749999999999995
--- Associated Data (2)---
	Mime_type: image/png
		foo.png ( obj1 )
	Mime_type: sina/curve
		internal ( timeplot_1, my_curves, my_other_curves )
--- Ensembles (0)---
	[]
--- Ensemble Attributes ---
--- Alias Feature Dictionary ---


### Associate Data to a Dataset

Associating data to a dataset means that you can reference that file and its data. The dataset doesn't store any of the data when you associate a file, it just references the file which means if the associated file is deleted, the data will no longer be "in" the dataset.

Below are a couple of common file formats and how to associate them to the correct `loader` through the `mime_type` so Kosh knows how to access the data. See [Example_02_Read_Data.ipynb](Example_02_Read_Data.ipynb) for more examples on how to associate data. 

If there is a file format that is not supported by Kosh, you can create your own custom loader [Example_Custom_Loader.ipynb](Example_Custom_Loader.ipynb).

In [5]:
# hdf5
dataset.associate("../tests/baselines/node_extracts2/node_extracts2.hdf5",
                  mime_type="hdf5",
                  metadata={"param10": "my value",
                            "my other param": "Example Text"},
                  absolute_path=False)

# csv
# The "pandas/*" mime types use the `pandas.read_*()` methods behind the scenes so you can pass in its arguments through `loader_kwargs`
dataset.associate("../tests/baselines/csv/my_csv_file.csv",
                  mime_type="pandas/csv",
                  metadata={"param20": "my other value",
                            "my param": 10},
                  loader_kwargs={'index_col': 0},
                  absolute_path=False)

# ultra
dataset.associate("my_ult_file.ult",
                  metadata={"param30": 45,
                            "my param": 560},
                  mime_type="ultra")

# These will show up in associated data with their correspondign `mime_type`
print(dataset)

KOSH DATASET
	id: obj1
	name: ???
	creator: ???

--- Attributes ---
	param1: 1
	param2: 2
	param3: 3.3
	param4: 1
	param5: test
	param6: 3.14
	some_variable_mean: 4.7749999999999995
--- Associated Data (5)---
	Mime_type: hdf5
		../tests/baselines/node_extracts2/node_extracts2.hdf5 ( 58c607dc29734683b30af3d817c2ecc7 )
	Mime_type: image/png
		foo.png ( obj1 )
	Mime_type: pandas/csv
		../tests/baselines/csv/my_csv_file.csv ( 4b86ae50a02a43df84c4a7d26efc561b )
	Mime_type: sina/curve
		internal ( timeplot_1, my_curves, my_other_curves )
	Mime_type: ultra
		/g/g20/moreno45/Projects/ASCAML/kosh/examples/my_ult_file.ult ( a058a1c585bb439a9ef317f47cbf87eb )
--- Ensembles (0)---
	[]
--- Ensemble Attributes ---
--- Alias Feature Dictionary ---


### Available Data

The commands below allow a user to see what data is available in a dataset

In [6]:
print('Attributes:') # can be single value or list
print('\t',dataset.list_attributes())
print('\n')
print('Features Sets:')  # only for time history data
print('\t',dataset.list_features())

# If there are a lot of files and features, use_cache=True can be used
# so that Kosh doesn't have to search through all the files again
# print('\t',dataset.list_features(use_cache=True))

Attributes:
	 ['id', 'param1', 'param2', 'param3', 'param4', 'param5', 'param6', 'some_variable_mean']


Features Sets:
	 ['my_curves', 'my_curves/some_variable', 'my_curves/time', 'my_other_curves', 'my_other_curves/time', 'timeplot_1', 'timeplot_1/feature_a', 'timeplot_1/feature_b', 'timeplot_1/time', 'cycles', 'direction', 'elements', 'node', 'node/metrics_0', 'node/metrics_1', 'node/metrics_10', 'node/metrics_11', 'node/metrics_12', 'node/metrics_2', 'node/metrics_3', 'node/metrics_4', 'node/metrics_5', 'node/metrics_6', 'node/metrics_7', 'node/metrics_8', 'node/metrics_9', 'zone', 'zone/metrics_0', 'zone/metrics_1', 'zone/metrics_2', 'zone/metrics_3', 'zone/metrics_4', 'id', 'name', 'creator', 'mynewattribute', 'myotherattribute', 'myparam10', 'myparam20', 'myparam30', 'myparam40', 'myparam50', 'myparam60', 'Gaussian (a: 5.0 w: 5.0 c: 0.0)', 'Gaussian (a: 5.0 w: 5.0 c: 50.0)', 'A + B', 'Straight Line (m: 0.125 b: -2.5 xmin: 60.0 xmax: 40.0)', 'a.y+numpy.random.normal(size=100)', '

### Accessing Data

Kosh knows which data is located in which file so all the user has to do is call the feature name to acquire that data. You can post-process this data and add it to the dataset through `setattr()` or `dataset.add_curve()`.

See [Example_05a_Transformers.ipynb](Example_05a_Transformers.ipynb), [Example_05b_Transformers-SKL.ipynb](Example_05b_Transformers-SKL.ipynb), and [Example_05b_Transformers-SKL.ipynb](Example_05b_Transformers-SKL.ipynb) on more ways to post process the data.

#### Whole File

In [7]:
# hdf5
associated_hdf5 = list(dataset.find(mime_type="hdf5"))[0]
h5_file = dataset.open(Id=associated_hdf5.id)
print('HDF5')
print(h5_file,'\n\n')

# csv
# The "pandas/*" mime types use the `pandas.read_*()` methods behind the scenes so you will get a `pandas.DataFrame()`
associated_csv_pandas = list(dataset.find(mime_type="pandas/csv"))[0]
df = dataset.open(Id=associated_csv_pandas.id)
print('CSV')
print(df,'\n\n')

# ultra
# returns a dictionary with curves
associated_ultra = list(dataset.find(mime_type="ultra"))[0]
ultra = dataset.open(Id=associated_ultra.id)
print('ULTRA')
print(ultra,'\n\n')

HDF5
<HDF5 file "node_extracts2.hdf5" (mode r)> 


CSV
                                  id          name  \
0   c40699ca067a4e29ba0f25470cf29e57   new_dataset   
1   21ef0cd592d24c5c8fcc89b066fb7418            15   
2   f970bd06a67a4956bd475fc48bd5a214             8   
3   3a79fc992f664a88a08151826f352cdf  new_dataset2   
4   6c35fb8c25034483af7031fada507062            16   
5   b634fce83dfd4e73b1f7f92c43b5ee1d             7   
6   1c14f05f44d846499b390fd435827f7d             2   
7   95d16b109ab64c6680af6f0bdc02aa00             9   
8   c0215e9c4080430da59979f71009c045            14   
9   b1e66c735ac6483d88fe041ab70dab2c             0   
10  4b77902d818a4980b8e55ad2106cd73e            21   
11  8554af7b60a34492a202a5f6fd468da1             5   
12  f05a324c3693494b8d0692ee6fd4b4bc             4   
13  ea0e89d9cb9a476d8f6dd683e8fc2666             1   
14  c941ee7ecd0a48b48ef6c3bad13c0216            11   
15  50677728c8dd4844b54f179ea23f09cf            10   
16  373fbc3883504fcb92c9b9c

#### Multiple Features

In [8]:
# hdf5
data1 = dataset[["node/metrics_5",'node/metrics_11']][:]
print('HDF5')
print(data1,'\n\n')

# csv
# The "pandas/*" mime types use the `pandas.read_*()` methods behind the scenes so you will get a `pandas.DataFrame()`
data2 = dataset[["creator", "myparam40"]][:]
print('CSV')
print(data2,'\n\n')

# ultra
# returns a dictionary with curves
data3 = dataset[['Gaussian (a: 5.0 w: 5.0 c: 0.0)', 'O + R']][:]
print('ULTRA')
print(data3,'\n\n')

HDF5
[<HDF5 dataset "metrics_5": shape (2, 18), type "<f4">, <HDF5 dataset "metrics_11": shape (2, 18), type "<f4">] 


CSV
                             creator  myparam40
0   9b7d60f394284459a1ae979bb0af019f        NaN
1   9b7d60f394284459a1ae979bb0af019f   0.950918
2   9b7d60f394284459a1ae979bb0af019f   2.270797
3   9b7d60f394284459a1ae979bb0af019f        NaN
4   9b7d60f394284459a1ae979bb0af019f   2.466016
5   9b7d60f394284459a1ae979bb0af019f   0.020335
6   9b7d60f394284459a1ae979bb0af019f   1.064911
7   9b7d60f394284459a1ae979bb0af019f   0.108305
8   9b7d60f394284459a1ae979bb0af019f   1.364071
9   9b7d60f394284459a1ae979bb0af019f   0.686090
10  9b7d60f394284459a1ae979bb0af019f   2.985315
11  9b7d60f394284459a1ae979bb0af019f   0.454322
12  9b7d60f394284459a1ae979bb0af019f   0.371130
13  9b7d60f394284459a1ae979bb0af019f   0.709315
14  9b7d60f394284459a1ae979bb0af019f   2.333544
15  9b7d60f394284459a1ae979bb0af019f   2.803816
16  9b7d60f394284459a1ae979bb0af019f   2.337639
17  9b7d60f3

#### Single Feature

In [9]:
# hdf5
data1 = dataset["node/metrics_5"][:]
print('HDF5')
print(data1,'\n\n')

# csv
# The "pandas/*" mime types use the `pandas.read_*()` methods behind the scenes so you will get a `pandas.DataFrame()`
data2 = dataset["creator"][:]
print('CSV')
print(data2,'\n\n')

# ultra
# returns a dictionary with curves
data3 = dataset['Gaussian (a: 5.0 w: 5.0 c: 0.0)'][:]
print('ULTRA')
print(data3,'\n\n')

HDF5
<HDF5 dataset "metrics_5": shape (2, 18), type "<f4"> 


CSV
                             creator
0   9b7d60f394284459a1ae979bb0af019f
1   9b7d60f394284459a1ae979bb0af019f
2   9b7d60f394284459a1ae979bb0af019f
3   9b7d60f394284459a1ae979bb0af019f
4   9b7d60f394284459a1ae979bb0af019f
5   9b7d60f394284459a1ae979bb0af019f
6   9b7d60f394284459a1ae979bb0af019f
7   9b7d60f394284459a1ae979bb0af019f
8   9b7d60f394284459a1ae979bb0af019f
9   9b7d60f394284459a1ae979bb0af019f
10  9b7d60f394284459a1ae979bb0af019f
11  9b7d60f394284459a1ae979bb0af019f
12  9b7d60f394284459a1ae979bb0af019f
13  9b7d60f394284459a1ae979bb0af019f
14  9b7d60f394284459a1ae979bb0af019f
15  9b7d60f394284459a1ae979bb0af019f
16  9b7d60f394284459a1ae979bb0af019f
17  9b7d60f394284459a1ae979bb0af019f
18  9b7d60f394284459a1ae979bb0af019f
19  9b7d60f394284459a1ae979bb0af019f
20  9b7d60f394284459a1ae979bb0af019f
21  9b7d60f394284459a1ae979bb0af019f
22  9b7d60f394284459a1ae979bb0af019f
23  9b7d60f394284459a1ae979bb0af019f
24  9b7d6

#### Describe Feature

In [10]:
# hdf5
print('HDF5')
print(dataset.describe_feature(Id=associated_hdf5.id, feature="node/metrics_5"),"\n\n")

# csv
# The "pandas/*" mime types use the `pandas.read_*()` methods behind the scenes so you will get `pandas.DataFrame.describe()`
print('CSV')
print(dataset.describe_feature(Id=associated_csv_pandas.id, feature="creator"),"\n\n")

# ultra
print('ULTRA')
print(dataset.describe_feature(Id=associated_ultra.id, feature='Gaussian (a: 5.0 w: 5.0 c: 0.0)'),"\n\n")

HDF5
{'size': (2, 18), 'format': 'hdf5', 'type': dtype('<f4'), 'dimensions': [{'name': 'cycles', 'first': 11, 'last': 8, 'length': 2}, {'name': 'elements', 'first': 17, 'last': 15, 'length': 18}]} 


CSV
count                                   25
unique                                   1
top       9b7d60f394284459a1ae979bb0af019f
freq                                    25
Name: creator, dtype: object 


ULTRA
{'name': 'Gaussian (a: 5.0 w: 5.0 c: 0.0)', 'size': 10, 'first_time': -15.0, 'last_time': 12.272727272727257, 'min': 0.0006170490204333978, 'max': 4.995410739193376, 'type': dtype('float64')} 




## Outer Loop

This outer loop groups the individual simulations into ensembles for organization purposes.

### Adding Dataset to Ensembles

Once there are a lot of simulations that have been completed and their datasets created, we can group them together.

See [Example_Ensembles.ipynb](Example_Ensembles.ipynb) for more information on ensembles.

In [11]:
# Try to see if it already exists
ensemble = list(store.find_ensembles(name="My Example Ensemble"))

if len(ensemble)==0: # create ensemble if doesn't exist
    ensemble = store.create_ensemble(name="My Example Ensemble",
                                    metadata={"root":"/root/path/for/ensemble",
                                            "project":"Example"})
else: # already exists
    ensemble = ensemble[0] # get first ensemble out of find results

# add this dataset to this ensemble
ensemble.add(dataset)

print('----- Ensemble -----')
print(ensemble,"\n\n") # This will display the ids' of the datasets in the ensemble

print('----- Dataset -----')
print(dataset) # This will now display the ensembles this dataset is a member off

----- Ensemble -----
KOSH ENSEMBLE
	id: be89c88f29534e6b9179d33b44aaef4c
	name: My Example Ensemble
	creator: moreno45

--- Attributes ---
	creator: moreno45
	name: My Example Ensemble
	project: Example
	root: /root/path/for/ensemble
--- Associated Data (0)---
--- Member Datasets (1)---
	['obj1'] 


----- Dataset -----
KOSH DATASET
	id: obj1
	name: ???
	creator: ???

--- Attributes ---
	param1: 1
	param2: 2
	param3: 3.3
	param4: 1
	param5: test
	param6: 3.14
	some_variable_mean: 4.7749999999999995
--- Associated Data (5)---
	Mime_type: hdf5
		../tests/baselines/node_extracts2/node_extracts2.hdf5 ( 58c607dc29734683b30af3d817c2ecc7 )
	Mime_type: image/png
		foo.png ( obj1 )
	Mime_type: pandas/csv
		../tests/baselines/csv/my_csv_file.csv ( 4b86ae50a02a43df84c4a7d26efc561b )
	Mime_type: sina/curve
		internal ( timeplot_1, my_curves, my_other_curves )
	Mime_type: ultra
		/g/g20/moreno45/Projects/ASCAML/kosh/examples/my_ult_file.ult ( a058a1c585bb439a9ef317f47cbf87eb )
--- Ensembles (1)---
	

### Adding Multiple Datasets to Ensembles with the same attributes and organizing with ensemble tags

Datasets also can be part of multiple ensembles and they can be further organized within a single ensemble using `ensemble_tags`. 

For example, say you want to add your train, validation, and test datasets to a single ensemble but need to organize them as such. Adding an attribute to the dataset would make that attribute the same across all ensembles but the train, validation, and test split is randomized for each ensemble. Adding an attribute to the ensemble would be at the ensemble level and thus you would need three ensembles one for train, validation, and test. `ensemble_tags` allow the user to organize the datasets within the ensemble.

We also use `inherit_attributes=False` so that the datasets and ensembles as well as the different ensemebles containing the same datasets can have the same attributes or else there will be a clash since the same attributes are seen.

**Note:** If a dataset was added to another ensemble using the default parameter `inherit_attributes=True` and the new ensemble and/or dataset attributes have the same name, there will be a conflict. In order to fix this you need to update the special ensemble tag `'INHERIT_ATTRIBUTES'` to `False` for that other dataset ensemble relation. This means that the dataset attributes will no longer be tied to that other ensemble so there will no longer be a conflict. If the dataset belongs to multiple ensembles with `inherit_attributes=True` (and there are attribute conflicts), this will need to be done for all those different ensembles: `dataset.add_ensemble_tags(ensemble_id, {'INHERIT_ATTRIBUTES': False})`

In [12]:
temp_datasets = []
for i in range(20):
    metadata = {"param1": random.randint(0, 1),
                "param2": random.randint(-10, 10),
                "param3": random.randint(-100, 100),
                "param4": random.randint(-1000, 1000),
                "param5": random.randint(-10000, 10000),
                "param6": random.randint(-100000, 100000),
                }

    temp_dataset = store.create(id=f"ds_{i}", metadata=metadata)
    temp_datasets.append(temp_dataset)

for i in range(10):
    ensemble = store.create_ensemble(id=f"ens_{i}",
                                    metadata={"root":f"/root/path/for/ensemble{i}/",
                                              "project":f"Example {i}"})
    for j, temp_ds in enumerate(temp_datasets):

        ensemble_tags = {}

        if j % 2 == 0:
            ensemble_tags["even_or_odd"] = "even"
        else:
            ensemble_tags["even_or_odd"] = "odd"

        if j <= 11:
            ensemble_tags["data_type"] = "train data"
        elif j <= 15:
            ensemble_tags["data_type"] = "validation data"
        else:
            ensemble_tags["data_type"] = "test data"

        ensemble.add(temp_ds, inherit_attributes=False, ensemble_tags=ensemble_tags)

print(ensemble)
print(temp_dataset)

KOSH ENSEMBLE
	id: ens_9
	name: Unnamed Ensemble
	creator: moreno45

--- Attributes ---
	creator: moreno45
	name: Unnamed Ensemble
	project: Example 9
	root: /root/path/for/ensemble9/
--- Associated Data (0)---
--- Member Datasets (20)---
	['ds_0', 'ds_1', 'ds_2', 'ds_3', 'ds_4', 'ds_5', 'ds_6', 'ds_7', 'ds_8', 'ds_9', 'ds_10', 'ds_11', 'ds_12', 'ds_13', 'ds_14', 'ds_15', 'ds_16', 'ds_17', 'ds_18', 'ds_19']
KOSH DATASET
	id: ds_19
	name: Unnamed Dataset
	creator: moreno45

--- Attributes ---
	creator: moreno45
	name: Unnamed Dataset
	param1: 0
	param2: 0
	param3: -20
	param4: -950
	param5: 4772
	param6: 77698
--- Associated Data (0)---
--- Ensembles (10)---
	['ens_0', 'ens_1', 'ens_2', 'ens_3', 'ens_4', 'ens_5', 'ens_6', 'ens_7', 'ens_8', 'ens_9']
--- Ensemble Attributes ---
	--- Ensemble ens_0 ---
		['project', 'root']
		--- Ensemble Tags ---
			['data_type', 'even_or_odd']
	--- Ensemble ens_1 ---
		['project', 'root']
		--- Ensemble Tags ---
			['data_type', 'even_or_odd']
	--- Ensem

### Finding Datasets within Ensembles
We can use `ensemble.find()` to narrow down the search to datasets only within the ensemble instead of searching the whole store with `store.find()`. We can filter by attributes like `store.find()` but we have the added benefit of filtering by `ensemble_tags`.

In [13]:
target_data = {'param1': 1,
               'param3': DataRange(min=0, max=100, max_inclusive=True)}
target_ensemble_tags = {"data_type": "train data"}
found_datasets =  list(ensemble.find_datasets(data=target_data, ensemble_tags=target_ensemble_tags))
for fd in found_datasets:
    setattr(fd, 'test_attr', 42) # setting new attribute for each of the found datasets
    print(fd)

/g/g20/moreno45/Projects/ASCAML/kosh/kosh/store.py:910: UserWarning: It is not recommended to use the find function by mixing keys and the reserved key `data`
  warnings.warn(


KOSH DATASET
	id: ds_0
	name: Unnamed Dataset
	creator: moreno45

--- Attributes ---
	creator: moreno45
	name: Unnamed Dataset
	param1: 1
	param2: -10
	param3: 93
	param4: 573
	param5: 4222
	param6: 91528
	test_attr: 42
--- Associated Data (0)---
--- Ensembles (10)---
	['ens_0', 'ens_1', 'ens_2', 'ens_3', 'ens_4', 'ens_5', 'ens_6', 'ens_7', 'ens_8', 'ens_9']
--- Ensemble Attributes ---
	--- Ensemble ens_0 ---
		['project', 'root']
		--- Ensemble Tags ---
			['data_type', 'even_or_odd']
	--- Ensemble ens_1 ---
		['project', 'root']
		--- Ensemble Tags ---
			['data_type', 'even_or_odd']
	--- Ensemble ens_2 ---
		['project', 'root']
		--- Ensemble Tags ---
			['data_type', 'even_or_odd']
	--- Ensemble ens_3 ---
		['project', 'root']
		--- Ensemble Tags ---
			['data_type', 'even_or_odd']
	--- Ensemble ens_4 ---
		['project', 'root']
		--- Ensemble Tags ---
			['data_type', 'even_or_odd']
	--- Ensemble ens_5 ---
		['project', 'root']
		--- Ensemble Tags ---
			['data_type', 'even_or_odd']

### Finding Datasets within whole Store

You can also filter datasets at the overall store level

In [14]:
target_data = {'param1': 1,
               'param3': DataRange(min=0, max=100, max_inclusive=True),
               'param6': DataRange(min=-1000, max=100000)}

found_datasets = list(store.find(data=target_data)) #list(store.find()) for all datasets

for fd in found_datasets:
    setattr(fd, 'total', 10) # setting new attribute for each of the found datasets
    print(fd)


KOSH DATASET
	id: obj1
	name: ???
	creator: ???

--- Attributes ---
	param1: 1
	param2: 2
	param3: 3.3
	param4: 1
	param5: test
	param6: 3.14
	some_variable_mean: 4.7749999999999995
	total: 10
--- Associated Data (5)---
	Mime_type: hdf5
		../tests/baselines/node_extracts2/node_extracts2.hdf5 ( 58c607dc29734683b30af3d817c2ecc7 )
	Mime_type: image/png
		foo.png ( obj1 )
	Mime_type: pandas/csv
		../tests/baselines/csv/my_csv_file.csv ( 4b86ae50a02a43df84c4a7d26efc561b )
	Mime_type: sina/curve
		internal ( timeplot_1, my_curves, my_other_curves )
	Mime_type: ultra
		/g/g20/moreno45/Projects/ASCAML/kosh/examples/my_ult_file.ult ( a058a1c585bb439a9ef317f47cbf87eb )
--- Ensembles (1)---
	['be89c88f29534e6b9179d33b44aaef4c']
--- Ensemble Attributes ---
	--- Ensemble be89c88f29534e6b9179d33b44aaef4c ---
		['project', 'root']
--- Alias Feature Dictionary ---
KOSH DATASET
	id: ds_0
	name: Unnamed Dataset
	creator: moreno45

--- Attributes ---
	creator: moreno45
	name: Unnamed Dataset
	param1: 1
	

### Converting `store.find()` method to Pandas DataFrame

You can also pass in the same arguments in the `store.find()` method to the  `store.to_dataframe()` method to get the attributes of the filtered datasets. By default, it will always include ['id', 'name', 'creator'].

In [15]:
# All datasets
df = store.to_dataframe()
print('All Datasets')
print(df,'\n\n')

# Filtered datasets
df = store.to_dataframe(data=target_data)
print('Filtered Datasets')
print(df,'\n\n')

# Specific columns
df = store.to_dataframe(data=target_data, data_columns=['param1', 'param6'])
print('Filtered Datasets with specific columns')
print(df,'\n\n')

All Datasets
       id             name                           creator  param1  param2  \
0    obj1             <NA>                              <NA>       1       2   
1    ds_7  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       0       3   
2    ds_4  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       0       3   
3   ds_11  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1       9   
4   ds_18  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1      -1   
5    ds_2  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1       2   
6    ds_3  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1      -5   
7    ds_1  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       0      -5   
8   ds_10  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1       7   
9   ds_15  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1      -8   
10  ds_19  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       0       0   
11  ds_17  Unnamed Dataset 

### Converting `dataset.find()` method to Pandas DataFrame

You can also pass in the same arguments in the `dataset.find()` method to the  `dataset.to_dataframe()` method to get the attributes of the filtered associated files. By default, it will always include ['id', 'mime_type', 'uri', 'associated'].

In [16]:
# All Associated Files
df = dataset.to_dataframe()
print('All Associated Files')
print(df,'\n\n')

# Filtered Associated Files
target_data = {'my param': 10}
df = dataset.to_dataframe(data=target_data)
print('Filtered Associated Files')
print(df,'\n\n')

# Specific columns
df = dataset.to_dataframe(data=target_data, data_columns=['param20'])
print('Filtered Associated Files with specific columns')
print(df,'\n\n')

All Associated Files
                                 id   mime_type  \
0  a058a1c585bb439a9ef317f47cbf87eb       ultra   
1                              obj1   some_type   
2  4b86ae50a02a43df84c4a7d26efc561b  pandas/csv   
3  58c607dc29734683b30af3d817c2ecc7        hdf5   

                                                 uri associated  \
0  /g/g20/moreno45/Projects/ASCAML/kosh/examples/...     [obj1]   
1                                               <NA>       <NA>   
2             ../tests/baselines/csv/my_csv_file.csv     [obj1]   
3  ../tests/baselines/node_extracts2/node_extract...     [obj1]   

                                            fast_sha     loader_kwargs  \
0  448a457f7344ece8c8be9b4ff383ab8258cb3401bc391e...                {}   
1                                               <NA>                {}   
2  6ae16fcd8a5bfc197d94451a64e3aa76b3cdce1f2af548...  {'index_col': 0}   
3  2c0f45d3ab840e47510a3fc1e463884de3765191cb3d07...                {}   

  my other param

### Converting `ensemble.find()` method to Pandas DataFrame

You can also pass in the same arguments in the `ensemble.find()` method to the  `ensemble.to_dataframe()` method to get the attributes of the filtered datasets within that specific ensemble. By default, it will always include ['id', 'name', 'creator'] and both the ensemble attributes and ensemble tags but they can be turned off.

In [17]:
# All Datasets in Ensemble
df = ensemble.to_dataframe()
print('All Datasets in Ensemble')
print(df,'\n\n')

# Filtered Datasets in Ensemble
target_data = {'param1': 1,
               'param3': DataRange(min=0, max=100, max_inclusive=True)}
target_ensemble_tags = {"data_type": "train data"}
df = ensemble.to_dataframe(data=target_data, ensemble_tags=target_ensemble_tags)
print('Filtered Datasets in Ensemble')
print(df,'\n\n')

# Specific columns without ensemble attributes or ensemble tags
df = ensemble.to_dataframe(data=target_data, ensemble_tags=target_ensemble_tags,
                           data_columns=['param20'],
                           include_ensemble_attributes=False, include_ensemble_tags=False)
print('Filtered Associated Files with specific columns')
print(df,'\n\n')

All Datasets in Ensemble
       id             name                           creator  param1  param2  \
0    ds_7  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       0       3   
1    ds_4  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       0       3   
2   ds_11  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1       9   
3   ds_18  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1      -1   
4    ds_2  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1       2   
5    ds_3  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1      -5   
6    ds_1  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       0      -5   
7   ds_10  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1       7   
8   ds_15  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       1      -8   
9   ds_19  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       0       0   
10  ds_17  Unnamed Dataset  9b7d60f394284459a1ae979bb0af019f       0       2   
11  ds_12  Unna

/g/g20/moreno45/Projects/ASCAML/kosh/kosh/store.py:910: UserWarning: It is not recommended to use the find function by mixing keys and the reserved key `data`
  warnings.warn(
/g/g20/moreno45/Projects/ASCAML/kosh/kosh/store.py:910: UserWarning: It is not recommended to use the find function by mixing keys and the reserved key `data`
  warnings.warn(


### Other Capabilities

#### Moving Datasets
If you want to move datasets around see [Example_07_Transferring_Datasets.ipynb](Example_07_Transferring_Datasets.ipynb) and [Example_Moving_Datasets.ipynb](Example_Moving_Datasets.ipynb).

#### Parallel Access to Kosh Store
If you are running a lot of simulations in parallel (e.g. through Maestro or Merlin) and need to access the Kosh store in parallel as well see [Example_ThreadSafe.ipynb](Example_ThreadSafe.ipynb).
